# Imports

In [34]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [35]:
keras.mixed_precision.set_global_policy('mixed_float16')

In [36]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm.notebook as tqdm
import transformers

In [37]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: True


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).

In [38]:
# class Tokenizer(transformers.GPT2Tokenizer):

#     def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
#         if token_ids_1 is None:
#             return [self.bos_token_id, *token_ids_0, self.eos_token_id]

#         return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]

In [39]:
tokenizer: transformers.PreTrainedTokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

In [40]:
assert isinstance(tokenizer.eos_token_id, int)
pad_token_id: int = tokenizer.eos_token_id + 1

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0

In [41]:
df = pd.read_csv("../../data/rnn_seq2seq/rus-eng.tsv", sep="\t", usecols=[1, 3], names=["ru", "en"])

In [42]:
df

,ru,en
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?
...,...,...
830313,Меня не поставили в известность.,I was not informed.
830314,Меня не поставили в известность.,I have not been informed.
830315,Она не поставила меня в известность.,She didn't inform me.
830316,Мне просто надо было увидеть это самому.,I just had to see this for myself.


In [43]:
df["ru"].str.len().describe()

count    830318.000000
mean         30.799881
std          15.759152
min           2.000000
25%          22.000000
50%          28.000000
75%          36.000000
max         841.000000
Name: ru, dtype: float64

In [44]:
df["en"].str.len().describe()

count    830318.000000
mean         31.370901
std          15.583791
min           3.000000
25%          23.000000
50%          29.000000
75%          37.000000
max        1127.000000
Name: en, dtype: float64

In [45]:
import pickle

In [46]:
TOKENIZE_NOW = False

In [47]:
if TOKENIZE_NOW:
    tokens_ru = [tokenizer.encode(text) for text in df["ru"]]
    with open("../../data/rnn_seq2seq/ru.pkl", "wb") as f:
        pickle.dump(tokens_ru, f, protocol=pickle.HIGHEST_PROTOCOL)

In [48]:
if TOKENIZE_NOW:
    tokens_en = [tokenizer.encode(text) for text in df["en"]]
    with open("../../data/rnn_seq2seq/en.pkl", "wb") as f:
            pickle.dump(tokens_en, f, protocol=pickle.HIGHEST_PROTOCOL)

In [49]:
with open("../../data/rnn_seq2seq/ru.pkl", "rb") as f:
    tokens_ru = pickle.load(f)

with open("../../data/rnn_seq2seq/en.pkl", "rb") as f:
    tokens_en = pickle.load(f)

In [50]:
tokens_ru_len = [len(tokens) for tokens in tokens_ru]
tokens_en_len = [len(tokens) for tokens in tokens_en]

In [51]:
np.percentile(tokens_ru_len, 99)

np.float64(88.0)

In [52]:
np.percentile(tokens_ru_len, 97)

np.float64(67.0)

In [53]:
np.percentile(tokens_en_len, 99)

np.float64(18.0)

In [54]:
class Seq2SeqDataset(keras.utils.PyDataset):
    def __init__(
        self,
        input_sequences: list[list[int]],
        max_seq_length_input: int,
        output_sequences: list[list[int]],
        max_seq_length_output: int,
        tokenizer: transformers.PreTrainedTokenizer,
        pad_token_id: int,
        batch_size: int = 128,
        shuffle: bool = True,
        rng: np.random.Generator | None = None,
        **kwargs,
    ):
        super().__init__(**kwargs)

        if rng is None:
            rng = np.random.default_rng()

        self.input = input_sequences
        self.max_seq_lenth_input = max_seq_length_input
        self.output = output_sequences
        self.max_seq_lenth_output = max_seq_length_output
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = rng
        self.tokenizer = tokenizer
        self.pad_token_id = pad_token_id
        self.indices = np.arange(len(input_sequences))

        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_indices = self.indices[start:end]

        encoder_input_tokens = [self.input[i] for i in batch_indices]
        decoder_input_tokens = [
            [tokenizer.bos_token_id] + self.output[i] for i in batch_indices
        ]
        decoder_output_tokens = [
            self.output[i] + [tokenizer.eos_token_id] for i in batch_indices
        ]

        encoder_input = keras.utils.pad_sequences(
            encoder_input_tokens,
            maxlen=self.max_seq_lenth_input,
            padding="post",
            value=pad_token_id,
        )
        decoder_input = keras.utils.pad_sequences(
            decoder_input_tokens,
            maxlen=self.max_seq_lenth_output,
            padding="post",
            value=pad_token_id,
        )
        decoder_output = keras.utils.pad_sequences(
            decoder_output_tokens,
            maxlen=self.max_seq_lenth_output,
            padding="post",
            value=pad_token_id,
        )

        mask = (decoder_output != pad_token_id).astype(np.float32)

        return (encoder_input, decoder_input), decoder_output, mask

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [55]:
train_ds = Seq2SeqDataset(tokens_ru, 67, tokens_en, 18, tokenizer, pad_token_id, batch_size=128)

In [56]:
val_translations = [
    ("Привет, как дела?", "Hi, how are you?"),
    ("Я люблю читать книги.", "I love reading books."),
    ("Сегодня хорошая погода.", "The weather is nice today."),
    ("Он не пришёл на встречу.", "He didn't come to the meeting."),
    ("Мы поедем в Москву завтра.", "We will go to Moscow tomorrow."),
    ("Она готовит ужин на кухне.", "She is cooking dinner in the kitchen."),
    ("Мне нравится этот фильм.", "I like this movie."),
    ("Дети играют во дворе.", "The children are playing in the yard."),
    ("Я не знаю, что делать.", "I don't know what to do."),
    ("Этот дом был построен в прошлом году.", "This house was built last year."),
]

In [57]:
val_ru_tokens = []
val_en_tokens = []

for ru, en in val_translations:
    val_ru_tokens.append(tokenizer.encode(ru))
    val_en_tokens.append(tokenizer.encode(en))

In [58]:
val_ds = Seq2SeqDataset(val_ru_tokens, 67, val_en_tokens, 18, tokenizer, pad_token_id, batch_size=10)

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.

In [59]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    rnn_type: type[keras.layers.Layer] = keras.layers.LSTM,
) -> keras.Model:
    """Creates a model with RNN architecture for sequence to sequence classification.

    Arguments:
        units: dimensionality of RNN cells
        n_tokens: number of tokens in the tokenizer dictionary
        n_labels: number of labels to be predicted
        n_stacks: number of RNN cells in the stack (1 -- no stacking)
        bidirectional: whether or not the model is bidirectional
        name: the model name
        cell_type: type of a cell to use, either keras.layers.LSTMCell or keras.layers.GRUCell

    Returns:
        The model"""

    enc_inputs = keras.Input(shape=(None,), dtype="int32", name="encoder_inputs")
    dec_inputs = keras.Input(shape=(None,), dtype="int32", name="decoder_inputs")

    enc_emb = keras.layers.Embedding(n_tokens, units, name="encoder_embedding")
    dec_emb = keras.layers.Embedding(n_tokens, units, name="decoder_embedding")

    x = enc_emb(enc_inputs)

    enc_states = None
    for i in range(n_stacks):
        is_last = i == n_stacks - 1
        layer_name = f"encoder_{i}"

        if bidirectional:
            rnn = keras.layers.Bidirectional(
                rnn_type(
                    units=units,
                    return_sequences=True,
                    return_state=True,
                ),
                name=layer_name,
            )
            x, *states = rnn(x)

            n_states = len(states) // 2
            enc_states = [
                keras.layers.Dense(
                    units,
                    activation="tanh",
                    name=f"enc_state_proj_{i}_{j}",
                )(
                    keras.layers.Concatenate(name=f"enc_state_concat_{i}_{j}")(
                        [states[j], states[n_states + j]]
                    )
                )
                for j in range(n_states)
            ]
        else:
            rnn = rnn_type(
                units=units,
                return_sequences=True,
                return_state=True,
                name=layer_name,
            )
            x, *enc_states = rnn(x)

    y = dec_emb(dec_inputs)

    for i in range(n_stacks):
        rnn = rnn_type(
            units=units,
            return_sequences=True,
            return_state=True,
            name=f"decoder_{i}",
        )
        if i == 0:
            y, *_ = rnn(y, initial_state=enc_states)
        else:
            y, *_ = rnn(y)

    logits = keras.layers.Dense(n_labels, name="output")(y)

    return keras.Model([enc_inputs, dec_inputs], logits, name=name)

Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

In [60]:
model = get_model(
    units=256,
    n_tokens=tokenizer.vocab_size + 1,
    n_labels=tokenizer.vocab_size + 1,
    n_stacks=1,
    bidirectional=False,
    name='seq2seq_lstm',
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=['accuracy'],
)

In [61]:
model.summary()

Model: "seq2seq_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 256) │ 12,866,048 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │ 12,866,048 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0 (LSTM)    │ [(None, None,     │    525,312 │ encoder_embeddin… │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_0 (LSTM)    │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ encoder_0[0][1],  │
│                     │ 256), (None,      │            │ encoder_0[0][2]   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, None,      │ 12,916,306 │ decoder_0[0][0]   │
│                     │ 50258)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 39,699,026 (151.44 MB)

 Trainable params: 39,699,026 (151.44 MB)

 Non-trainable params: 0 (0.00 B)

# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one. 

In [72]:
from pathlib import Path

In [ ]:
LOGS_DIR = Path('../../logs/rnn_seq2seq')
CHECKPOINT_DIR = Path('../../model_checkpoints/rnn_seq2seq')

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [63]:
from datetime import datetime

In [64]:
from torch.utils.tensorboard import SummaryWriter


class PyTorchTensorBoard(keras.callbacks.Callback):
    def __init__(self, log_dir):
        super().__init__()
        self.writer = SummaryWriter(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        if logs:
            for key, value in logs.items():
                self.writer.add_scalar(key, value, epoch)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_DIR / f"model_train_accuracy_{datetime.now().strftime('%Y%m%d-%H%M%S')}.keras",
        monitor="accuracy",
        mode="max",
        save_best_only=True,
    ),
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_DIR / f"model_val_accuracy_{datetime.now().strftime('%Y%m%d-%H%M%S')}.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
    ),
    PyTorchTensorBoard(LOGS_DIR),
]

In [67]:
model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)

Epoch 1/5


c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:651: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, (h_n, c_n) = torch._VF.lstm(


6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1207s 186ms/step - accuracy: 0.4464 - loss: 1.5692 - val_accuracy: 0.7179 - val_loss: 0.6412
Epoch 2/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1451s 224ms/step - accuracy: 0.6117 - loss: 0.9902 - val_accuracy: 0.8205 - val_loss: 0.3410
Epoch 3/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 2020s 311ms/step - accuracy: 0.6833 - loss: 0.7583 - val_accuracy: 0.8590 - val_loss: 0.2357
Epoch 4/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1878s 290ms/step - accuracy: 0.7212 - loss: 0.6396 - val_accuracy: 0.8205 - val_loss: 0.2313
Epoch 5/5
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1675s 258ms/step - accuracy: 0.7454 - loss: 0.5661 - val_accuracy: 0.8846 - val_loss: 0.1786


# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).

In [68]:
def build_inference_models(model, cell_type=keras.layers.LSTM):
    n_states = 2 if cell_type is keras.layers.LSTM else 1

    enc_emb = model.get_layer('encoder_embedding')
    dec_emb = model.get_layer('decoder_embedding')
    enc_lstm = model.get_layer('encoder_0')
    dec_lstm = model.get_layer('decoder_0')
    out_dense = model.get_layer('output')

    units = enc_lstm.units

    enc_in = keras.Input(shape=(None,), dtype='int32', name='enc_in_inf')
    x = enc_emb(enc_in)
    _, *enc_states = enc_lstm(x)
    encoder_inf = keras.Model(enc_in, enc_states, name='encoder_inf')

    dec_in = keras.Input(shape=(1,), dtype='int32', name='dec_in_inf')
    state_ins = [
        keras.Input(shape=(units,), name=f'state_in_{i}')
        for i in range(n_states)
    ]

    y = dec_emb(dec_in)
    y, *new_states = dec_lstm(y, initial_state=state_ins)
    logits = out_dense(y)

    decoder_inf = keras.Model(
        [dec_in] + state_ins,
        [logits] + new_states,
        name='decoder_inf',
    )

    return encoder_inf, decoder_inf

In [69]:
encoder_inf, decoder_inf = build_inference_models(model, cell_type=keras.layers.LSTM)

In [70]:
def translate(
    text, tokenizer, encoder_inf, decoder_inf, max_len=50, return_tokens=False
):
    bos = tokenizer.bos_token_id
    eos = tokenizer.eos_token_id
    pad = tokenizer.pad_token_id

    enc_ids = tokenizer(text, add_special_tokens=False, return_tensors="np")[
        "input_ids"
    ]

    states = encoder_inf.predict(enc_ids, verbose=0)
    if not isinstance(states, list):
        states = [states]

    tokens = []
    next_id = bos

    for _ in range(max_len):
        dec_in = np.array([[next_id]], dtype="int32")
        outputs = decoder_inf.predict([dec_in] + states, verbose=0)
        logits, *states = outputs

        next_id = int(np.argmax(logits[0, -1, :]))

        if next_id == eos:
            break
        if next_id == pad:
            break

        tokens.append(next_id)

    if return_tokens:
        return tokens, tokenizer.decode(tokens, skip_special_tokens=True)

    return tokenizer.decode(tokens, skip_special_tokens=True)

In [74]:
for ru, en in val_translations:
    pred = translate(ru, tokenizer, encoder_inf, decoder_inf)
    print(f"RU  : {ru}")
    print(f"REF : {en}")
    print(f"PRED: {pred}")
    print("-" * 60)

RU  : Привет, как дела?
REF : Hi, how are you?
PRED: Do you know how to get a haircut?
------------------------------------------------------------
RU  : Я люблю читать книги.
REF : I love reading books.
PRED: I like reading books.
------------------------------------------------------------
RU  : Сегодня хорошая погода.
REF : The weather is nice today.
PRED: This weather is good today.
------------------------------------------------------------
RU  : Он не пришёл на встречу.
REF : He didn't come to the meeting.
PRED: He didn't come to the meeting.
------------------------------------------------------------
RU  : Мы поедем в Москву завтра.
REF : We will go to Moscow tomorrow.
PRED: We're going to go to school.
------------------------------------------------------------
RU  : Она готовит ужин на кухне.
REF : She is cooking dinner in the kitchen.
PRED: She is preparing dinner for dinner.
------------------------------------------------------------
RU  : Мне нравится этот фильм.
REF : 

In [76]:
df[df["ru"].str.contains("Москва")]

,ru,en
1626,Москва не сразу строилась.,Rome wasn't built in a day.
1627,Москва не сразу строилась.,Rome was not built in a day.
21953,Тебе нравится Москва?,Do you like Moscow?
36995,Москва - российский город.,Moscow is a Russian town.
36996,Москва — столица России.,Moscow is the capital of Russia.
38755,Вам нравится Москва?,Do you like Moscow?
157586,Москва смотрится красиво из космоса.,Moscow looks beautiful from space.
226277,"Москва — единственный в мире город, расположен...","Moscow is the only city in the world, located ..."
228836,Разве столица России — это не Москва?,Is the capital city of Russia not Moscow?
228837,Разве столица России не Москва?,Is the capital city of Russia not Moscow?


In [77]:
df[df["ru"].str.contains("школа")]

,ru,en
332,Эта школа была основана в 1650 году.,The school was established in 1650.
333,Эта школа была основана в 1650 году.,This school was founded in 1650.
1202,Здание справа — это школа.,The building on the right side is a school.
2535,Наша школа находится за рекой.,Our school is across the river.
3898,Это не Центральная школа вождения?,Isn't this Central Driving School?
...,...,...
779587,Завтра школа.,It's a school night.
779941,В каком году открылась эта школа?,What year did this school open?
782094,Тут раньше школа была.,There used to be a school here.
783248,Новая школа тебе больше нравится?,Do you like your new school better?


In [78]:
df[df["ru"].str.contains("кухня")]

,ru,en
3132,Мне нравится корейская кухня из-за острых специй.,I love Korean food because of the hot spices.
11468,В некоторых молочных кухнях раздают бесплатные...,Some hospitals hand out free samples of baby m...
24718,"Её кухня укомплектована устройствами, облегчаю...",Her kitchen is equipped with labor-saving devi...
35468,Мне нравится корейская кухня.,I like Korean cuisine.
39423,Ему нравится итальянская кухня.,He likes Italian food.
...,...,...
786920,Тебе не нравится китайская кухня?,Don't you like Chinese food?
809951,Какая красивая кухня!,What a beautiful kitchen!
810663,Тайская кухня пользуется большой популярностью...,Thai food is extremely popular in the United S...
822532,"Он спросил, нравится ли мне китайская кухня.",He asked if I liked Chinese food.


In [79]:
df[df["ru"].str.contains("двор")]

,ru,en
2336,Во дворце живут король с королевой.,In the palace live the king and the queen.
4525,Он содержит небольшой птичий двор.,He keeps a small poultry yard.
13920,В этом дворце живут король и королева.,This is the palace the king and queen live in.
17612,Ребята играют в мяч на заднем дворе.,The boys are throwing a ball in the back yard.
18513,Хелен играет во дворе.,Helen is playing in the yard.
...,...,...
817218,По двору расхаживали куры и гуси.,Chickens and geese were walking around the yard.
822823,"Это дворец, в котором живут король с королевой.",This is the palace the king and queen live in.
822825,"Это и есть дворец, в котором живут король с ко...",This is the palace the king and queen live in.
824683,"Боюсь, как бы меня не выдворили из страны.",I'm afraid of being kicked out of the country.


In [84]:
df[df["ru"].str.contains("Привет, как дела?")]

,ru,en
9220,"Привет, как дела?","Hello, how are you?"
9221,"Привет, как дела?","Hi, how's it going?"
9222,"Привет, как дела?","Hi, how are you?"
9223,"Привет, как дела?",Hi. How are you doing?
9224,"Привет, как дела?","Hi, how do you do?"
397674,"Привет, как дела?",Hi! How are you?
397681,"Привет, как дела?",Hi. How are you?
397686,"Привет, как дела?",Hello! How are you?
397690,"Привет, как дела?","Hello, how are you doing?"
637621,"«Привет, как дела?» – «Всё хорошо».","Hi! How are you? ""I'm doing well."""


In [93]:
df[df["en"].str.contains("Do you know how to get a haircut?")]

,ru,en


In [95]:
df[df["en"].str.contains("haircut")]

,ru,en
2615,Скоро мне придётся постричься.,Soon I have to go get a haircut.
2916,Я подстригаюсь каждый месяц.,I get a haircut every month.
18885,Меня пугают люди с причёской «Ирокез»!,People with Mohawk haircuts frighten me!
42734,Тому надо подстричься.,Tom needs a haircut.
52395,Тебе нравится моя новая причёска?,Do you like my new haircut?
...,...,...
828950,"Том ушёл в парикмахерскую, но скоро должен вер...","Tom has gone to get a haircut, but he should b..."
830256,Тебе давно пора стричься.,It's high time you had a haircut.
830257,Тебе давно пора постричься.,It's high time you had a haircut.
830258,Вам давно пора постричься.,It's high time you had a haircut.


In [97]:
translate("Ты знаешь как подстричься?", tokenizer, encoder_inf, decoder_inf)

c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:651: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, (h_n, c_n) = torch._VF.lstm(


'Do you know how to get to the beach?'

In [98]:
translate("Ты знаешь как пройти на пляж?", tokenizer, encoder_inf, decoder_inf)

'Do you know how to ride a horse?'

In [99]:
translate("Ты знаешь как ездить на лошади?", tokenizer, encoder_inf, decoder_inf)

'Do you know how to ride a horse?'

In [100]:
df[df["en"].str.contains("horse")]

,ru,en
2213,"Ты не думаешь, что моя лошадь победит в Дерби?",Don't you think my horse ought to win the Derby?
4566,Дарёному коню в зубы не смотрят.,Don't look a gift horse in the mouth.
4567,Дарёному коню в зубы не смотрят.,Never look a gift horse in the mouth.
4859,"Если скрестить лошадь с ослом, то получится мул.",If you mate a horse with an ass you will get a...
6290,"Слепой лошади что кивай, что подмигивай.",A nod is as good as a wink to a blind horse.
...,...,...
825989,Купи хрена.,Buy some horseradish.
826005,Я нашёл лошадь.,I found the horse.
826006,Лошадь я нашёл.,I found the horse.
829747,"Человек, который говорит правду, нуждается в б...",A man who speaks the truth needs a fast horse.


In [101]:
df[df["en"].str.contains("beach")]

,ru,en
1970,В какой стороне пляж?,Which way is the beach?
3715,Волна за волной накатывались на пляж.,Wave after wave surged upon the beach.
7670,Много часов она провалялась на пляже.,She idled away many hours on the beach.
9866,"Так как я не на службе сегодня, пошли на пляж.","As I am off duty today, let's go to the beach."
10823,Они ушли на пляж.,They went to the beach.
...,...,...
827266,Не хочешь ли пойти со мной на пляж?,Would you like to go to the beach with me?
827267,Не хотите ли пойти со мной на пляж?,Would you like to go to the beach with me?
827268,Ты хотел бы пойти со мной на пляж?,Would you like to go to the beach with me?
827269,Вы хотели бы пойти со мной на пляж?,Would you like to go to the beach with me?


In [102]:
model.fit(train_ds, validation_data=val_ds, epochs=10, initial_epoch=5, callbacks=callbacks)

Epoch 6/10


c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:651: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, (h_n, c_n) = torch._VF.lstm(


6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1228s 189ms/step - accuracy: 0.7627 - loss: 0.5148 - val_accuracy: 0.8846 - val_loss: 0.1729
Epoch 7/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1238s 191ms/step - accuracy: 0.7761 - loss: 0.4764 - val_accuracy: 0.8590 - val_loss: 0.1595
Epoch 8/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1235s 190ms/step - accuracy: 0.7868 - loss: 0.4463 - val_accuracy: 0.8718 - val_loss: 0.1512
Epoch 9/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1207s 186ms/step - accuracy: 0.7956 - loss: 0.4223 - val_accuracy: 0.8718 - val_loss: 0.1631
Epoch 10/10
6487/6487 ━━━━━━━━━━━━━━━━━━━━ 1242s 191ms/step - accuracy: 0.8031 - loss: 0.4022 - val_accuracy: 0.8846 - val_loss: 0.1495


In [103]:
encoder_inf, decoder_inf = build_inference_models(model, cell_type=keras.layers.LSTM)

In [104]:
for ru, en in val_translations:
    pred = translate(ru, tokenizer, encoder_inf, decoder_inf)
    print(f"RU  : {ru}")
    print(f"REF : {en}")
    print(f"PRED: {pred}")
    print("-" * 60)

RU  : Привет, как дела?
REF : Hi, how are you?
PRED: Do you know how to get a radio?
------------------------------------------------------------
RU  : Я люблю читать книги.
REF : I love reading books.
PRED: I like reading books.
------------------------------------------------------------
RU  : Сегодня хорошая погода.
REF : The weather is nice today.
PRED: Today is good weather.
------------------------------------------------------------
RU  : Он не пришёл на встречу.
REF : He didn't come to the meeting.
PRED: He didn't come to the meeting.
------------------------------------------------------------
RU  : Мы поедем в Москву завтра.
REF : We will go to Moscow tomorrow.
PRED: We'll go to school.
------------------------------------------------------------
RU  : Она готовит ужин на кухне.
REF : She is cooking dinner in the kitchen.
PRED: She is preparing for dinner.
------------------------------------------------------------
RU  : Мне нравится этот фильм.
REF : I like this movie.
PRED